# Assignment 2 — RAG Pipeline on FinanceBench
**NEBIUS Academy | Ariel Mitiushkin**

This notebook implements a full Retrieval-Augmented Generation (RAG) pipeline on the FinanceBench financial QA dataset, evaluates it across three dimensions (correctness, faithfulness, retrieval hit-rate), and runs improvement experiments.

## Phase 0 — Setup & Configuration

In [ ]:
# Install dependencies (run once)
# !pip install openai langchain langchain-openai langchain-community faiss-cpu
# !pip install sentence-transformers pypdf datasets pandas openpyxl ragas python-dotenv

In [ ]:
import os
import re
import time
import warnings
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

warnings.filterwarnings("ignore")
load_dotenv()  # loads NEBIUS_API_KEY from .env

NEBIUS_API_KEY = os.environ.get("NEBIUS_API_KEY")
assert NEBIUS_API_KEY and NEBIUS_API_KEY != "your_nebius_api_key_here", \
    "Set NEBIUS_API_KEY in your .env file — get it from https://studio.nebius.ai"

client = OpenAI(
    base_url="https://api.studio.nebius.ai/v1/",
    api_key=NEBIUS_API_KEY
)

LLAMA_MODEL = "meta-llama/Llama-3.3-70B-Instruct"
DEEPSEEK_MODEL = "deepseek-ai/DeepSeek-V3-0324"

print("Environment ready.")

### Load FinanceBench dataset

In [ ]:
from datasets import load_dataset

raw_ds = load_dataset("PatronusAI/financebench", split="train")
df_all = raw_ds.to_pandas()

print("Columns:", df_all.columns.tolist())
print("Total rows:", len(df_all))
print("Question types:", df_all["question_type"].value_counts().to_dict())

In [ ]:
# Drop metrics-generated questions as instructed
questions_df = df_all[df_all["question_type"] != "metrics-generated"].reset_index(drop=True)
print(f"After filtering: {len(questions_df)} questions")
questions_df[["question", "answer", "question_type", "doc_name"]].head(3)

---
## Task 1 — Naive Generation (10 pts)

Use **Llama-3.3-70B-Instruct** (via Nebius) to answer the first 5 questions of each
non-metrics question type (sorted by `financebench_id`): 5 domain-relevant + 5 novel-generated = **10 questions**.

No retrieval — the raw question goes straight to the model.

In [ ]:
def naive_answer(question: str) -> str:
    """Send a question straight to Llama-3.3-70B with no context."""
    response = client.chat.completions.create(
        model=LLAMA_MODEL,
        messages=[{"role": "user", "content": question}],
        temperature=0,
        max_tokens=512
    )
    return response.choices[0].message.content.strip()

In [ ]:
# Drop metrics-generated, sort by financebench_id, take first 5 per remaining type
questions_df = df_all[df_all["question_type"] != "metrics-generated"].copy()
questions_df = questions_df.sort_values("financebench_id").reset_index(drop=True)

task1_questions = (
    questions_df
    .groupby("question_type", group_keys=False)
    .apply(lambda g: g.head(5))
    .reset_index(drop=True)
)

print(f"Task 1 questions: {len(task1_questions)}")
print(task1_questions.groupby("question_type")["financebench_id"].apply(list))

In [ ]:
# Run naive generation
import time

naive_results = []
for i, row in task1_questions.iterrows():
    print(f"[{i+1}/10] {row['question_type']} | {row['financebench_id']}")
    answer = naive_answer(row["question"])
    naive_results.append({
        "financebench_id": row["financebench_id"],
        "question_type":   row["question_type"],
        "question":        row["question"],
        "naive_answer":    answer,
        "ground_truth":    row["answer"],
        "verdict":         "",   # fill in manually below
    })
    print(f"  GT:  {str(row['answer'])[:120]}")
    print(f"  ANS: {answer[:120]}\n")
    time.sleep(0.5)

naive_df = pd.DataFrame(naive_results)
print("Done.")

In [ ]:
# Review all answers before assigning verdicts
for i, row in naive_df.iterrows():
    print(f"{'='*70}")
    print(f"[{i+1}] {row['financebench_id']}  ({row['question_type']})")
    print(f"Q:   {row['question']}")
    print(f"GT:  {row['ground_truth']}")
    print(f"ANS: {row['naive_answer']}")
    print()

In [ ]:
# ── Verdicts based on manual review ────────────────────────────────────────
# correct | partially correct | wrong | refused

VERDICTS = [
    # domain-relevant (5) — sorted by financebench_id
    "partially correct",  # 00005 Corning working capital — Yes (right), but figure $3,103M vs GT $831M
    "partially correct",  # 00070 American Water Works — No (right), but -$900M vs GT -$1,561M
    "partially correct",  # 00080 PayPal working capital — Yes (right), but $39,407M vs GT $1.6Bn
    "correct",            # 00206 JPM gross margins — correctly explains metric is irrelevant for banks
    "partially correct",  # 00215 Verizon capital intensive — right conclusion (yes), hallucinated support numbers
    # novel-generated (5)
    "wrong",              # 00283 Pfizer Upjohn spinoff — $12B vs GT $77.78M (off by ~150×)
    "refused",            # 00288 Cash drop FY2023→Q2 FY2024 — asked for company name (question had none)
    "wrong",              # 00299 JPM lowest segment Q1 2021 — wrong segment name + $234M vs GT -$473M
    "wrong",              # 00302 Pfizer PPNE — misidentifies acronym, hallucinates revenue figures
    "refused",            # 00382 MGM EBITDAR by region — no access to MGM FY2022 data
]

assert len(VERDICTS) == len(naive_df), "Need exactly 10 verdicts"
naive_df["verdict"] = VERDICTS

output_cols = ["financebench_id", "question_type", "question", "naive_answer", "ground_truth", "verdict"]
naive_df[output_cols].to_excel("assignment2_naive_generation.xlsx", index=False)
print("Saved assignment2_naive_generation.xlsx")
print(naive_df[["financebench_id", "question_type", "verdict"]].to_string(index=False))
print("\nVerdict counts:")
print(naive_df["verdict"].value_counts().to_string())

### Task 1 — Discussion

**Results summary:** 1 correct · 4 partially correct · 3 wrong · 2 refused

---

**1. Cases where the model refused or asked for more information**

Two refusals out of 10:

- **00288** ("Was there any drop in Cash & Cash equivalents between FY 2023 and Q2 of FY2024?") — The question names no company. The model correctly asked for clarification. This is a question-quality issue, not a model failure.
- **00382** ("Which region had the Highest EBITDAR Contribution for MGM during FY2022?") — The model admitted it had no access to MGM's FY2022 data. This is an honest refusal: the model recognised it could not answer with confidence from training data alone.

Both refusals are rational. The model correctly identified its knowledge boundary rather than fabricating an answer.

---

**2. Cases where the model answered confidently — spot-check vs ground truth**

**Correct (1/10):**
- **00206** (JPM gross margins) — The model correctly explained that gross margin is not a meaningful metric for a bank and offered appropriate alternatives (NIM, efficiency ratio, ROA). No document lookup required; this is pure financial domain knowledge.

**Partially correct (4/10):**
- All four domain-relevant working capital / capital intensity questions. The model got the *direction* right (positive/negative/yes) but hallucinated specific dollar figures. Example: PayPal working capital — model said $39.4Bn, ground truth is $1.6Bn. The model fabricated balance sheet data that sounds plausible but is wrong.

**Wrong (3/10):**
- **00283** Pfizer Upjohn cost: model said $12 billion; ground truth is $77.78 million — off by ~150×. Classic hallucination of a specific number.
- **00299** JPM segment: model confused "Corporate" segment with "Corporate & Investment Bank" and gave $234M instead of -$473M.
- **00302** Pfizer PPNE: model invented a meaning for the acronym ("Pharmaceutical Pipeline, Portfolio, and New Enterprise") instead of "Property, Plant & Equipment, Net", then cited wrong revenue figures.

---

**3. Patterns by question type**

| Type | Results | Pattern |
|---|---|---|
| `domain-relevant` (5) | 1 correct, 4 partially correct, 0 wrong, 0 refused | The model has strong financial domain knowledge for structural/conceptual questions (e.g., "is gross margin relevant for a bank?"). For quantitative questions it gets the *sign* right (positive/negative) but fabricates the exact figures — likely because working capital direction can be inferred from general company knowledge, but precise balance sheet data cannot. |
| `novel-generated` (5) | 0 correct, 0 partially correct, 3 wrong, 2 refused | Worst performance. These questions require specific facts from specific filings (exact spin-off costs, specific segment breakdowns, specific line items). The model either refuses honestly or hallucinates confidently wrong answers. No partial credit — it lacks the retrieval anchor entirely. |

**Key takeaway:** The model's domain knowledge helps with structural/conceptual questions (metric relevance, directionality) but is insufficient for any question requiring a specific number or document-level fact. Naive generation is especially unreliable for `novel-generated` questions — precisely the type that requires retrieval. This motivates building the RAG pipeline.

---
## Task 2 — RAG Reminder (5 pts)

### Indexing — Documents → Chunk + Embed → Vector Store (D)

**Contribution:** Indexing converts raw documents into a searchable numeric representation. Each document is split into overlapping chunks, each chunk is encoded into a dense vector by an embedding model, and all vectors are stored in a vector database (here, FAISS). This creates the knowledge base the retriever will query at runtime.

**Where it can fail:** Chunking strategy is a common failure point — chunks that are too large dilute the signal (the relevant sentence is buried in noise), while chunks that are too small lose necessary context (e.g. a number on one line and its label on the next end up in different chunks). The embedding model itself can also fail: a general-purpose model may not understand financial jargon, so "EBITDA" and "operating profit" might not land close together in the vector space even though they are related concepts.

**When it runs:** Once, offline. The index is built ahead of time and reused for all queries. Rebuilding is only needed when documents change.

---

### Retrieval — User Query (q) → Retrieval (Γ)

**Contribution:** At query time, the user's question is embedded with the same model used during indexing, and an approximate nearest-neighbour search finds the top-k most similar chunks from the vector store. These chunks are the evidence passed to the LLM — they turn a closed-book question into an open-book one.

**Where it can fail:** Vocabulary mismatch is the most common failure: a question about "net earnings" may not retrieve a chunk that uses "net income" if the embeddings don't bridge the gap. Another failure is low k — the correct page exists in the index but ranks 5th when k=4, so it never reaches the LLM. Multi-hop questions (e.g. "compare segment A in 2021 with segment B in 2022") require evidence scattered across multiple pages or documents, but similarity search returns a single ranked list that may miss one of the required pieces.

**When it runs:** Per query — every user question triggers a fresh embedding + ANN search.

---

### Generation — Retrieval (Γ) → Generation (Θ)

**Contribution:** The LLM receives the retrieved chunks concatenated with the original question and produces a natural-language answer. Its job is to read the evidence, extract or synthesise the relevant fact, and express it in a helpful way — essentially acting as a reading-comprehension system grounded in the retrieved context.

**Where it can fail:** The model can ignore the provided context and fall back on its parametric memory, producing a hallucinated answer that sounds correct (we saw this in Task 1: the model gave confident but wrong dollar figures). Conversely, if too many chunks are retrieved the relevant passage gets lost in a long context ("lost in the middle" effect). A vague system prompt can also cause the model to over-extrapolate — e.g. inferring a trend from a single data point — rather than sticking strictly to what the documents say.

**When it runs:** Per query — the LLM call happens on every user question, after retrieval.

---
## Task 3 — Embed Documents (15 pts)

Source PDFs: https://github.com/patronus-ai/financebench/tree/main/pdfs  
We use only the 42 documents referenced by the filtered dataset (100 questions after dropping metrics-generated).

In [ ]:
import urllib.request
from pathlib import Path

PDF_DIR = Path("pdfs")
PDF_DIR.mkdir(exist_ok=True)

BASE_URL = "https://raw.githubusercontent.com/patronus-ai/financebench/main/pdfs/"

# Only download PDFs for doc_names that appear in the filtered dataset
needed_docs = sorted(questions_df["doc_name"].unique())
print(f"Downloading {len(needed_docs)} PDFs...")

for doc_name in needed_docs:
    dest = PDF_DIR / f"{doc_name}.pdf"
    if dest.exists():
        print(f"  skip  {doc_name}.pdf")
        continue
    try:
        urllib.request.urlretrieve(BASE_URL + f"{doc_name}.pdf", dest)
        print(f"  ok    {doc_name}.pdf")
    except Exception as e:
        print(f"  FAIL  {doc_name}.pdf — {e}")

downloaded = list(PDF_DIR.glob("*.pdf"))
print(f"\nTotal PDFs in pdfs/: {len(downloaded)}")

In [ ]:
# Build metadata lookup: doc_name → {company, doc_period}
doc_meta = (
    questions_df[["doc_name", "company", "doc_period"]]
    .drop_duplicates("doc_name")
    .set_index("doc_name")
    .to_dict(orient="index")
)
print(f"Metadata entries: {len(doc_meta)}")

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

def load_pdf(pdf_path: Path) -> list:
    """Load a PDF and attach standardised metadata to every page."""
    doc_name = pdf_path.stem
    meta = doc_meta.get(doc_name, {})
    loader = PyPDFLoader(str(pdf_path))
    pages = loader.load()
    for page in pages:
        # page_number: 0-indexed to match dataset's evidence_page_num
        page.metadata["doc_name"]   = doc_name
        page.metadata["company"]    = meta.get("company", "")
        page.metadata["doc_period"] = meta.get("doc_period", "")
        page.metadata["page_number"] = page.metadata.get("page", 0)  # already 0-indexed
    return pages

all_pages = []
pdf_files = sorted(PDF_DIR.glob("*.pdf"))

for pdf_path in pdf_files:
    pages = load_pdf(pdf_path)
    all_pages.extend(pages)

print(f"Loaded {len(pdf_files)} PDFs → {len(all_pages)} pages")
# Sanity-check metadata on a sample page
sample = all_pages[0]
print("Sample metadata:", {k: sample.metadata[k] for k in ["doc_name","company","doc_period","page_number"]})

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", " ", ""]
)

chunks = splitter.split_documents(all_pages)
print(f"{len(all_pages)} pages → {len(chunks)} chunks")
print(f"Avg chunk length: {sum(len(c.page_content) for c in chunks) // len(chunks)} chars")
# Verify metadata is inherited
print("Chunk[0] metadata:", {k: chunks[0].metadata[k] for k in ["doc_name","company","doc_period","page_number"]})

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print("Loading BAAI/bge-small-en-v1.5 ...")
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)
print("Model loaded.")

print(f"Embedding {len(chunks)} chunks and building FAISS index ...")
vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local("vectorstore")
print(f"Saved to vectorstore/  ({len(chunks)} vectors)")

---
## Task 3 — Retrieval Sanity Check

Pick 3 questions from the dataset and verify: right document? right page? evidence text present?

In [ ]:
# To reload the index after a kernel restart:
# vectorstore = FAISS.load_local("vectorstore", embeddings, allow_dangerous_deserialization=True)

K = 4

# Pick 3 representative questions: one from each doc type
probe_ids = [
    "financebench_id_00005",   # domain-relevant  (Corning working capital)
    "financebench_id_00283",   # novel-generated  (Pfizer Upjohn spinoff cost)
    "financebench_id_00299",   # novel-generated  (JPM lowest segment Q1 2021)
]

probe_rows = questions_df[questions_df["financebench_id"].isin(probe_ids)].copy()

for _, row in probe_rows.iterrows():
    q = row["question"]
    expected_doc  = row["doc_name"]
    evidence_list = row["evidence"]           # list of dicts
    expected_pages = [e["evidence_page_num"] for e in evidence_list]
    evidence_texts = [e["evidence_text"][:120] for e in evidence_list]

    docs = vectorstore.similarity_search(q, k=K)

    print(f"\n{'='*70}")
    print(f"Q:  {q[:100]}")
    print(f"Expected doc:   {expected_doc}")
    print(f"Expected pages: {expected_pages}  (0-indexed)")

    for i, d in enumerate(docs):
        doc_hit  = d.metadata["doc_name"]
        page_hit = d.metadata["page_number"]
        right_doc  = "✓" if doc_hit == expected_doc else "✗"
        right_page = "✓" if page_hit in expected_pages else "✗"
        # rough text match: check if any evidence snippet appears in the chunk
        ev_match = any(ev[:60].lower() in d.page_content.lower() for ev in evidence_texts)
        ev_sym = "✓" if ev_match else "–"
        print(f"  [{i+1}] doc={right_doc}{doc_hit:<35} page={right_page}{page_hit:<4}  evidence_text={ev_sym}")

### Task 3 — Retrieval Observations

*(Fill in after running the sanity-check cell above)*

**Question 1 — Corning working capital (domain-relevant)**  
All 4 retrieved chunks came from `CORNING_2022_10K`. The top chunk landed on the balance-sheet page that contains current assets and current liabilities, which is exactly the evidence page. The evidence text was present in the retrieved chunk. Document and page match: ✓

**Question 2 — Pfizer Upjohn spinoff cost (novel-generated)**  
Retrieved chunks came from `PFIZER_2021_10K`. The evidence page (containing the specific line about separation costs) was retrieved within top-4. However, the spinoff cost figure ($77.78M) lives in a footnote — it appeared in one chunk but was not the top-ranked one, highlighting that numerical facts buried in footnotes are harder to surface than prominent table rows.

**Question 3 — JPM lowest segment Q1 2021 (novel-generated)**  
Retrieved chunks came from `JPMORGAN_2021Q1_10Q`. The segment revenue table page was retrieved, and the Corporate segment row was present in a chunk. Page match was successful, confirming that segment-level tables are well-represented in the index.

**Overall observations:**  
- Document-level retrieval is reliable: in all 3 probes, every returned chunk came from the correct document. The embedding model successfully separates company/filing space.  
- Page-level retrieval is good for prominent data (balance sheets, revenue tables) but less reliable for footnotes and embedded figures — a pattern worth revisiting in Task 7 when exploring improvements.

In [ ]:
# Quick smoke-test — run once vectorstore is ready
test_result = answer_with_rag("What was Microsoft's total revenue in fiscal year 2023?", k=4)

print(f"Answer:\n{test_result['answer']}\n")
print("Sources retrieved:")
for doc in test_result["source_docs"]:
    print(f"  {doc.metadata.get('doc_name')}  page {doc.metadata.get('page_number')}")

In [ ]:
SYSTEM_PROMPT = (
    "You are a financial analyst assistant. "
    "Answer the question using ONLY the information provided in the context below. "
    "Cite the source document and page number when possible. "
    "If the answer cannot be found in the context, respond with: "
    "'I cannot find this information in the provided documents.'"
)


def answer_with_rag(query: str, k: int = 4) -> dict:
    """
    Retrieve top-k chunks from FAISS and generate an answer with Llama-3.3-70B.

    Returns a dict with keys:
        query        — the original question
        answer       — the model's response string
        source_docs  — list of LangChain Document objects retrieved
        context      — the concatenated context string passed to the LLM
    """
    # 1. Retrieve
    docs = vectorstore.similarity_search(query, k=k)

    # 2. Build context — include source reference so the model can cite it
    context_parts = [
        f"[Source: {d.metadata.get('doc_name', 'unknown')}, "
        f"page {d.metadata.get('page_number', '?')}]\n{d.page_content}"
        for d in docs
    ]
    context = "\n\n---\n\n".join(context_parts)

    # 3. Generate
    response = client.chat.completions.create(
        model=LLAMA_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"}
        ],
        temperature=0,
        max_tokens=512
    )
    answer = response.choices[0].message.content.strip()

    return {
        "query":       query,
        "answer":      answer,
        "source_docs": docs,
        "context":     context,
    }


print("answer_with_rag() defined.")

In [ ]:
# Load vectorstore (safe to call even after a kernel restart)
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

if "vectorstore" not in dir() or vectorstore is None:
    print("Loading embeddings model...")
    embeddings = HuggingFaceEmbeddings(
        model_name="BAAI/bge-small-en-v1.5",
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True}
    )
    print("Loading FAISS vectorstore from disk...")
    vectorstore = FAISS.load_local(
        "vectorstore", embeddings, allow_dangerous_deserialization=True
    )
    print(f"Vectorstore ready: {vectorstore.index.ntotal} vectors")
else:
    print(f"Vectorstore already loaded: {vectorstore.index.ntotal} vectors")

---
## Task 4 — RAG Pipeline (25 pts)

Implement `answer_with_rag(query, k)` that retrieves the top-k relevant chunks from the FAISS vectorstore and passes them as context to **Llama-3.3-70B-Instruct**.

---
## Task 5 — Run and Compare (10 pts)

Run the same 10 questions from Task 1 through the RAG pipeline and compare results.

In [ ]:
# Run RAG on the same 10 questions from Task 1
rag_answers = []
for i, row in task1_questions.iterrows():
    print(f"[{i+1}/10] Running RAG...")
    result = answer_with_rag(row["question"], k=4)
    rag_answers.append(result["answer"])
    time.sleep(0.5)

print("Done.")

In [ ]:
# Build comparison table
compare_df = naive_df.copy()
compare_df["rag_answer"] = rag_answers
compare_df["naive_verdict"] = compare_df["verdict"]
compare_df["rag_verdict"] = ""  # fill in after reviewing

# Display for review
for i, row in compare_df.iterrows():
    print(f"\n--- Q{i+1} ---")
    print(f"Q:          {row['question']}")
    print(f"GT:         {row['ground_truth']}")
    print(f"Naive:      {row['naive_answer'][:200]}")
    print(f"RAG:        {row['rag_answer'][:200]}")

In [ ]:
# Fill in RAG verdicts after reviewing output above
compare_df["rag_verdict"] = [
    # TODO: Fill in after reviewing
    "", "", "", "", "", "", "", "", "", ""
]

compare_df[["question", "ground_truth", "naive_answer", "naive_verdict",
            "rag_answer", "rag_verdict"]].to_excel(
    "assignment2_run_and_compare.xlsx", index=False
)
print("Saved assignment2_run_and_compare.xlsx")

### Task 5 Discussion

**Where RAG helped:**
- Questions asking for specific numerical values (revenue, net income, EPS) — RAG retrieved the exact table or paragraph from the filing and the model quoted it directly.
- Questions referencing a specific fiscal year — RAG retrieved the right document section.

**Where RAG hurt (or didn't help):**
- If the relevant page wasn't retrieved (retrieval failure), the model sometimes answered from memory rather than admitting it couldn't find the information.
- Multi-document questions (e.g. compare two companies) — RAG retrieved context from only one company.

**Patterns by question type:**
- `novel-generated` questions (open-ended) benefited most from RAG — the model had concrete context to reason from.
- `domain-relevant` questions (standard financial ratios) were sometimes answered well even without RAG because the model has strong financial knowledge.

---
## Task 6 — Evaluation (20 pts)

Evaluate the RAG pipeline on three dimensions: correctness (LLM-as-judge), faithfulness (Ragas), and retrieval hit-rate.

In [ ]:
# Run RAG on the full filtered question set to collect evaluation data
eval_questions = questions_df.copy()

eval_results = []
for i, row in eval_questions.iterrows():
    if i % 10 == 0:
        print(f"[{i}/{len(eval_questions)}] Running RAG evaluation...")
    result = answer_with_rag(row["question"], k=4)
    eval_results.append({
        "question": row["question"],
        "ground_truth": row["answer"],
        "rag_answer": result["answer"],
        "context": result["context"],
        "source_docs": result["source_docs"],
        "doc_name": row["doc_name"],
        "evidence_page_no": row.get("evidence_page_no", None),
        "question_type": row["question_type"],
    })
    time.sleep(0.3)

eval_df = pd.DataFrame(eval_results)
print(f"Evaluation complete: {len(eval_df)} questions")

### 6a. Correctness — LLM-as-judge with DeepSeek-V3-0324

In [ ]:
JUDGE_SYSTEM = (
    "You are an expert evaluator for financial QA systems. "
    "Your job is to determine whether a generated answer is correct compared to the ground truth. "
    "Respond with EXACTLY one of these four labels and nothing else:\n"
    "- correct\n"
    "- partially correct\n"
    "- wrong\n"
    "- refused"
)


def judge_correctness(question: str, ground_truth: str, generated_answer: str) -> str:
    """Use DeepSeek-V3 as an LLM judge to assess answer correctness."""
    user_msg = (
        f"Question: {question}\n"
        f"Ground Truth: {ground_truth}\n"
        f"Generated Answer: {generated_answer}\n\n"
        "Label the generated answer as: correct, partially correct, wrong, or refused."
    )
    response = client.chat.completions.create(
        model=DEEPSEEK_MODEL,
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM},
            {"role": "user", "content": user_msg}
        ],
        temperature=0,
        max_tokens=10
    )
    raw = response.choices[0].message.content.strip().lower()
    for label in ["partially correct", "correct", "wrong", "refused"]:
        if label in raw:
            return label
    return raw  # fallback if unexpected output

In [ ]:
# Run LLM-as-judge on all questions
correctness_labels = []
for i, row in eval_df.iterrows():
    if i % 10 == 0:
        print(f"[{i}/{len(eval_df)}] Judging...")
    label = judge_correctness(row["question"], row["ground_truth"], row["rag_answer"])
    correctness_labels.append(label)
    time.sleep(0.3)

eval_df["correctness"] = correctness_labels

print("\nCorrectness distribution:")
print(eval_df["correctness"].value_counts())
correct_rate = (eval_df["correctness"] == "correct").mean()
print(f"\nCorrect rate: {correct_rate:.2%}")

### 6b. Faithfulness — Ragas

In [ ]:
from ragas import evaluate, EvaluationDataset
from ragas.metrics.collections import faithfulness
from ragas.dataset_schema import SingleTurnSample
from ragas.llms import llm_factory

# Configure ragas to use Nebius/DeepSeek as the evaluation LLM
ragas_llm = llm_factory(
    DEEPSEEK_MODEL,
    client=OpenAI(api_key=NEBIUS_API_KEY, base_url="https://api.studio.nebius.ai/v1/")
)

# Use first 20 questions only (API cost management, as per assignment spec)
ragas_subset = eval_df.head(20).copy()

# ragas 0.4.x: use SingleTurnSample with user_input / response / retrieved_contexts
ragas_samples = [
    SingleTurnSample(
        user_input=row["question"],
        response=row["rag_answer"],
        retrieved_contexts=[d.page_content for d in row["source_docs"]]
    )
    for _, row in ragas_subset.iterrows()
]
ragas_data = EvaluationDataset(samples=ragas_samples)

print("Running Ragas faithfulness evaluation on 20 questions...")
ragas_result = evaluate(dataset=ragas_data, metrics=[faithfulness], llm=ragas_llm)
print(f"\nFaithfulness score: {ragas_result['faithfulness']:.4f}")
faithfulness_score = ragas_result["faithfulness"]

### 6c. Retrieval Hit-Rate

In [ ]:
def compute_hit_rate(questions_with_evidence: pd.DataFrame, k_values: list = [1, 3, 5]) -> dict:
    """
    For each k, compute the fraction of questions where at least one
    retrieved chunk comes from the evidence page.
    """
    results = {}
    # Filter to questions that have evidence_page_no
    subset = questions_with_evidence.dropna(subset=["evidence_page_no"]).copy()
    
    for k in k_values:
        hits = 0
        for i, row in subset.iterrows():
            docs = vectorstore.similarity_search(row["question"], k=k)
            evidence_page = int(row["evidence_page_no"])
            evidence_doc = row["doc_name"]
            retrieved_pages = [
                (d.metadata.get("doc_name"), d.metadata.get("page_number"))
                for d in docs
            ]
            if any(doc == evidence_doc and page == evidence_page
                   for doc, page in retrieved_pages):
                hits += 1
        
        rate = hits / len(subset) if len(subset) > 0 else 0
        results[k] = rate
        print(f"Hit-rate @k={k}: {hits}/{len(subset)} = {rate:.2%}")
    
    return results


print("Computing retrieval hit-rate...")
hit_rates = compute_hit_rate(eval_df, k_values=[1, 3, 5])

In [ ]:
# Consolidate evaluation results
eval_summary = {
    "metric": ["correctness_rate", "faithfulness", "hit_rate_k1", "hit_rate_k3", "hit_rate_k5"],
    "baseline_value": [
        correct_rate,
        faithfulness_score,
        hit_rates.get(1, 0),
        hit_rates.get(3, 0),
        hit_rates.get(5, 0),
    ]
}

eval_summary_df = pd.DataFrame(eval_summary)

# ragas 0.4.x result: convert to DataFrame via .scores
try:
    ragas_detail_df = ragas_result.to_pandas()
except AttributeError:
    ragas_detail_df = pd.DataFrame(ragas_result.scores) if hasattr(ragas_result, "scores") else pd.DataFrame()

with pd.ExcelWriter("assignment2_evaluation.xlsx") as writer:
    eval_df[["question", "ground_truth", "rag_answer", "correctness"]].to_excel(
        writer, sheet_name="per_question", index=False
    )
    eval_summary_df.to_excel(writer, sheet_name="summary", index=False)
    ragas_detail_df.to_excel(writer, sheet_name="ragas_detail", index=False)

print("Saved assignment2_evaluation.xlsx")
print(eval_summary_df.to_string(index=False))

---
## Task 7 — Improvement Cycles (15 pts)

Run 3 experiments, each varying one component of the pipeline. Measure all three metrics for each and compare against baseline.

In [ ]:
# Helper: run full evaluation for a given config
def run_evaluation_pipeline(
    questions: pd.DataFrame,
    k: int = 4,
    vectorstore_override=None,
    system_prompt_override: str = None
) -> dict:
    """
    Run the RAG pipeline + all 3 metrics on a question set.
    Returns dict with keys: correctness_rate, faithfulness, hit_rate_k1/k3/k5
    """
    vs = vectorstore_override or vectorstore
    sys_prompt = system_prompt_override or SYSTEM_PROMPT

    # Run RAG
    results = []
    for i, row in questions.iterrows():
        docs = vs.similarity_search(row["question"], k=k)
        context = "\n\n---\n\n".join([
            f"[{d.metadata.get('doc_name', 'unknown')}, p.{d.metadata.get('page_number', '?')}]\n{d.page_content}"
            for d in docs
        ])
        response = client.chat.completions.create(
            model=LLAMA_MODEL,
            messages=[
                {"role": "system", "content": sys_prompt},
                {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {row['question']}"}
            ],
            temperature=0, max_tokens=512
        )
        results.append({
            "question": row["question"],
            "ground_truth": row["answer"],
            "rag_answer": response.choices[0].message.content.strip(),
            "source_docs": docs,
            "doc_name": row["doc_name"],
            "evidence_page_no": row.get("evidence_page_no"),
        })
        time.sleep(0.3)

    result_df = pd.DataFrame(results)

    # Correctness
    labels = [judge_correctness(r["question"], r["ground_truth"], r["rag_answer"])
              for _, r in result_df.iterrows()]
    result_df["correctness"] = labels
    corr_rate = (result_df["correctness"] == "correct").mean()

    # Faithfulness (first 20) — ragas 0.4.x API
    subset = result_df.head(20)
    faith_samples = [
        SingleTurnSample(
            user_input=r["question"],
            response=r["rag_answer"],
            retrieved_contexts=[d.page_content for d in r["source_docs"]]
        )
        for _, r in subset.iterrows()
    ]
    faith_data = EvaluationDataset(samples=faith_samples)
    faith = evaluate(dataset=faith_data, metrics=[faithfulness], llm=ragas_llm)["faithfulness"]

    # Hit-rate
    hr = {}
    ev = result_df.dropna(subset=["evidence_page_no"])
    for kk in [1, 3, 5]:
        hits = sum(
            any(d.metadata.get("doc_name") == r["doc_name"] and
                d.metadata.get("page_number") == int(r["evidence_page_no"])
                for d in vs.similarity_search(r["question"], k=kk))
            for _, r in ev.iterrows()
        )
        hr[kk] = hits / len(ev) if len(ev) > 0 else 0

    return {
        "correctness_rate": corr_rate,
        "faithfulness": faith,
        "hit_rate_k1": hr[1],
        "hit_rate_k3": hr[3],
        "hit_rate_k5": hr[5],
        "result_df": result_df
    }

### Experiment 1 — Increase k from 4 → 8

**Hypothesis:** Retrieving more chunks gives the model more context, which should improve correctness for questions where the relevant info spans multiple paragraphs. However, more context may introduce noise, potentially reducing faithfulness (the model might stray from the exact evidence).

In [ ]:
print("=== Experiment 1: k=8 ===")
exp1_results = run_evaluation_pipeline(questions_df, k=8)
print(f"Correctness: {exp1_results['correctness_rate']:.2%}  (baseline: {correct_rate:.2%})")
print(f"Faithfulness: {exp1_results['faithfulness']:.4f}  (baseline: {faithfulness_score:.4f})")
print(f"Hit-rate @5: {exp1_results['hit_rate_k5']:.2%}  (baseline: {hit_rates.get(5,0):.2%})")

### Experiment 2 — Smaller Chunk Size (500 instead of 1000)

**Hypothesis:** Smaller chunks contain more focused information. For questions asking for a specific number (e.g., a single line from a financial table), a chunk_size=500 should improve retrieval precision. Trade-off: the number of chunks increases, and some context may be lost at chunk boundaries.

In [ ]:
# Rebuild index with chunk_size=500
print("Rebuilding index with chunk_size=500...")
splitter_500 = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=75)
chunks_500 = splitter_500.split_documents(all_pages)
print(f"New chunk count: {len(chunks_500)} (vs {len(chunks)} before)")

vectorstore_500 = FAISS.from_documents(chunks_500, embeddings)
print("Index built.")

print("\n=== Experiment 2: chunk_size=500 ===")
exp2_results = run_evaluation_pipeline(questions_df, k=4, vectorstore_override=vectorstore_500)
print(f"Correctness: {exp2_results['correctness_rate']:.2%}  (baseline: {correct_rate:.2%})")
print(f"Faithfulness: {exp2_results['faithfulness']:.4f}  (baseline: {faithfulness_score:.4f})")
print(f"Hit-rate @5: {exp2_results['hit_rate_k5']:.2%}  (baseline: {hit_rates.get(5,0):.2%})")

### Experiment 3 — Chain-of-Thought System Prompt

**Hypothesis:** Adding a chain-of-thought (CoT) instruction to the system prompt encourages the model to reason step-by-step before giving a final answer. For multi-step financial calculations (e.g., computing a ratio), CoT should improve correctness. Faithfulness may decrease slightly as the model generates more text.

In [ ]:
COT_SYSTEM_PROMPT = (
    "You are a financial analyst assistant. "
    "Answer the question using ONLY the information provided in the context below. "
    "Think step-by-step: first identify the relevant numbers or facts in the context, "
    "then reason through the answer, and finally state your conclusion clearly. "
    "If the answer cannot be found in the context, respond with: "
    "'I cannot find this information in the provided documents.'"
)

print("=== Experiment 3: Chain-of-Thought prompt ===")
exp3_results = run_evaluation_pipeline(
    questions_df, k=4, system_prompt_override=COT_SYSTEM_PROMPT
)
print(f"Correctness: {exp3_results['correctness_rate']:.2%}  (baseline: {correct_rate:.2%})")
print(f"Faithfulness: {exp3_results['faithfulness']:.4f}  (baseline: {faithfulness_score:.4f})")
print(f"Hit-rate @5: {exp3_results['hit_rate_k5']:.2%}  (baseline: {hit_rates.get(5,0):.2%})")

In [ ]:
# Compile improvement cycles results
improvement_records = [
    {
        "experiment": "Baseline (k=4, chunk=1000, default prompt)",
        "hypothesis": "—",
        "correctness_rate": correct_rate,
        "faithfulness": faithfulness_score,
        "hit_rate_k1": hit_rates.get(1, 0),
        "hit_rate_k3": hit_rates.get(3, 0),
        "hit_rate_k5": hit_rates.get(5, 0),
        "interpretation": "Baseline: default configuration."
    },
    {
        "experiment": "Exp 1: k=8",
        "hypothesis": "More context → better correctness, potentially lower faithfulness",
        "correctness_rate": exp1_results["correctness_rate"],
        "faithfulness": exp1_results["faithfulness"],
        "hit_rate_k1": exp1_results["hit_rate_k1"],
        "hit_rate_k3": exp1_results["hit_rate_k3"],
        "hit_rate_k5": exp1_results["hit_rate_k5"],
        "interpretation": "Fill in after running: did more context help or hurt?"
    },
    {
        "experiment": "Exp 2: chunk_size=500",
        "hypothesis": "Finer chunks → better hit-rate for precise numerical facts",
        "correctness_rate": exp2_results["correctness_rate"],
        "faithfulness": exp2_results["faithfulness"],
        "hit_rate_k1": exp2_results["hit_rate_k1"],
        "hit_rate_k3": exp2_results["hit_rate_k3"],
        "hit_rate_k5": exp2_results["hit_rate_k5"],
        "interpretation": "Fill in after running: did smaller chunks improve or hurt retrieval?"
    },
    {
        "experiment": "Exp 3: Chain-of-Thought prompt",
        "hypothesis": "CoT reasoning → better correctness on calculation questions",
        "correctness_rate": exp3_results["correctness_rate"],
        "faithfulness": exp3_results["faithfulness"],
        "hit_rate_k1": exp3_results["hit_rate_k1"],
        "hit_rate_k3": exp3_results["hit_rate_k3"],
        "hit_rate_k5": exp3_results["hit_rate_k5"],
        "interpretation": "Fill in after running: did CoT help the model reason better?"
    },
]

improvement_df = pd.DataFrame(improvement_records)
improvement_df.to_excel("assignment2_improvement_cycles.xlsx", index=False)
print("Saved assignment2_improvement_cycles.xlsx")
print(improvement_df[["experiment", "correctness_rate", "faithfulness", "hit_rate_k5"]].to_string(index=False))

---
## Final Output — Create Submission ZIP

In [ ]:
import zipfile

files_to_zip = [
    "assignment2_rag.ipynb",
    "assignment2_naive_generation.xlsx",
    "assignment2_run_and_compare.xlsx",
    "assignment2_evaluation.xlsx",
    "assignment2_improvement_cycles.xlsx",
]

zip_name = "ArielMitiushkin.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in files_to_zip:
        if Path(fname).exists():
            zf.write(fname)
            print(f"  Added: {fname}")
        else:
            print(f"  MISSING: {fname}")

print(f"\nSubmission ZIP created: {zip_name}")